In [1]:
from bin_picking.objects.objects import register_stl_body, StlBody, MojocoEnv, Table, Box
import mujoco
import mujoco.viewer
import time

In [2]:
mj = MojocoEnv("test_model", [StlBody("bin").at(0.2, 0.1, 1.5),StlBody("bin").at(0.1, 0.3, 1.6), StlBody("bin").at(0.1, 0.4, 1.5)])
mj.worldbody.append(Table())
mj.worldbody.append(Box().at(0, 0, 1))

In [3]:
xml = mj.xml_spec()

In [4]:
print(xml)

<mujoco model="test_model">
  <worldbody>
    <geom name="floor" type="plane" size="5 5 0.1" rgba="0.2 0.3 0.4 1" />
    <body name="bin_0" pos="0.2 0.1 1.5">
      <geom type="mesh" mesh="bin_part_0" material="body_material" friction="1 0.005 0.0001" />
      <geom type="mesh" mesh="bin_part_1" material="body_material" friction="1 0.005 0.0001" />
      <geom type="mesh" mesh="bin_part_2" material="body_material" friction="1 0.005 0.0001" />
      <geom type="mesh" mesh="bin_part_3" material="body_material" friction="1 0.005 0.0001" />
      <geom type="mesh" mesh="bin_part_4" material="body_material" friction="1 0.005 0.0001" />
      <geom type="mesh" mesh="bin_part_5" material="body_material" friction="1 0.005 0.0001" />
      <geom type="mesh" mesh="bin_part_6" material="body_material" friction="1 0.005 0.0001" />
      <geom type="mesh" mesh="bin_part_7" material="body_material" friction="1 0.005 0.0001" />
      <geom type="mesh" mesh="bin_part_8" material="body_material" fricti

In [ ]:
m = mujoco.MjModel.from_xml_string(xml)
d = mujoco.MjData(m)

with mujoco.viewer.launch_passive(m, d) as viewer:
  # Close the viewer automatically after 30 wall-seconds.
  start = time.time()
  while viewer.is_running() and time.time() - start < 10:
    step_start = time.time()

    # mj_step can be replaced with code that also evaluates
    # a policy and applies a control signal before stepping the physics.
    mujoco.mj_step(m, d)

    # Example modification of a viewer option: toggle contact points every two seconds.
    with viewer.lock():
      viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = int(d.time % 2)

    # Pick up changes to the physics state, apply perturbations, update options from GUI.
    viewer.sync()

    # Rudimentary time keeping, will drift relative to wall clock.
    time_until_next_step = m.opt.timestep - (time.time() - step_start)
    if time_until_next_step > 0:
      time.sleep(time_until_next_step)

In [ ]:
import trimesh


stl_objects = mj.get_stl_objects()
for o in stl_objects:
    pos = d.body(o.name).xpos
    quat = d.body(o.name).xquat
    print(o.name, pos, quat)

bin_0 [0.40557179 0.08497534 0.8421214 ] [3.45588066e-06 1.56677828e-01 9.87649765e-01 3.48764211e-06]
bin_1 [0.09575042 0.3811742  0.96030802] [ 0.72238842 -0.68811953 -0.04761783  0.04877515]
bin_2 [0.10241408 0.3917311  0.81735977] [ 9.99927419e-01  1.60918951e-04  3.80535595e-04 -1.20410113e-02]


In [7]:
d